## GSAT trend patterns

In [2]:
# In[1]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [3]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster()

### Input from the FigureS2 noise pattern calculation

In [5]:
models = ['CanESM5', 'CESM2', 'IPSL_CM6A', 'EC_Earth3', 'ACCESS', 'MPI_ESM', 'MIROC6']
from pathlib import Path
base_dir = Path("/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3")

forced_ds = {}  # dict: model -> Dataset

for m in models:
    subdir = "SMILE_forced" if m != "CESM2" else "SMILE_force"
    dir_forced_input = base_dir / m / subdir
    if m == "CESM2":
        fn = dir_forced_input / "CESM2_CMIP6+smbb_ENSmean_forced_MK_trend_1950-2022_sliding.nc"
    else:
        fn = dir_forced_input / f"{m}_ENSmean_forced_MK_trend_1950-2022_sliding.nc"
    print(f"Opening {fn}")
    forced_ds[m] = xr.open_dataset(fn)

Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/CanESM5/SMILE_forced/CanESM5_ENSmean_forced_MK_trend_1950-2022_sliding.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/CESM2/SMILE_force/CESM2_CMIP6+smbb_ENSmean_forced_MK_trend_1950-2022_sliding.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/IPSL_CM6A/SMILE_forced/IPSL_CM6A_ENSmean_forced_MK_trend_1950-2022_sliding.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/EC_Earth3/SMILE_forced/EC_Earth3_ENSmean_forced_MK_trend_1950-2022_sliding.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/ACCESS/SMILE_forced/ACCESS_ENSmean_forced_MK_trend_1950-2022_sliding.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MPI_ESM/SMILE_forced/MPI_ESM_ENSmean_forced_MK_trend_1950-2022_sliding.nc
Opening /work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MIROC6/SMILE_forced/MIROC6_ENSmean_forced_MK_trend_1950-2022_sliding.nc


In [7]:
forced_ds["CESM2"]

<xarray.Dataset>
Dimensions:  (lon: 180, lat: 90, period: 64)
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
  * period   (period) object '1950-2022' '1951-2022' ... '2012-2022' '2013-2022'
Data variables:
    trend    (period, lat, lon) float64 ...
    p_value  (period, lat, lon) float64 ...

In [6]:
forced_ds["ACCESS"]

<xarray.Dataset>
Dimensions:  (lon: 180, lat: 90, period: 64)
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
    height   float64 ...
  * period   (period) object '1950-2022' '1951-2022' ... '2012-2022' '2013-2022'
Data variables:
    trend    (period, lat, lon) float64 ...
    p_value  (period, lat, lon) float64 ...

In [8]:
# MMEM trend can be calculated by averaging the trend from all models
forced_trend_da = xr.concat([forced_ds['CanESM5'].trend,forced_ds['IPSL_CM6A'].trend,forced_ds['CESM2'].trend,
                         forced_ds['EC_Earth3'].trend,forced_ds['ACCESS'].trend,
                         forced_ds['MPI_ESM'].trend,forced_ds['MIROC6'].trend],dim='model', coords='minimal')
MMEM_forced_trend_da = forced_trend_da.mean(dim='model')

In [9]:
forced_trend_da

<xarray.DataArray 'trend' (model: 7, period: 64, lat: 90, lon: 180)>
array([[[[ 3.27537594e-01,  3.27532134e-01,  3.28951763e-01, ...,
           3.23894290e-01,  3.25174675e-01,  3.27091257e-01],
         [ 3.26388687e-01,  3.28775511e-01,  3.30552971e-01, ...,
           3.21929686e-01,  3.22985292e-01,  3.24964360e-01],
         [ 3.27250824e-01,  3.30951610e-01,  3.33968474e-01, ...,
           3.21315922e-01,  3.22780027e-01,  3.23768380e-01],
         ...,
         [ 7.75315916e-01,  7.76888132e-01,  7.76985662e-01, ...,
           7.68303997e-01,  7.70505339e-01,  7.71435533e-01],
         [ 7.45054725e-01,  7.46734698e-01,  7.47484917e-01, ...,
           7.41024065e-01,  7.43037707e-01,  7.44088900e-01],
         [ 7.37772599e-01,  7.37795137e-01,  7.37506304e-01, ...,
           7.33433331e-01,  7.34207204e-01,  7.35300518e-01]],

        [[ 3.34127715e-01,  3.34129425e-01,  3.35065326e-01, ...,
           3.30616294e-01,  3.31515046e-01,  3.32679622e-01],
         [ 3.34663902e-01,  3.36474213e-01,  3.38139795e-01, ...,
           3.30425666e-01,  3.31843199e-01,  3.33128587e-01],
         [ 3.35292915e-01,  3.36510841e-01,  3.39678323e-01, ...,
           3.28634455e-01,  3.29700080e-01,  3.32155048e-01],
...
         [ 5.33959717e-01,  5.29959053e-01,  5.26374280e-01, ...,
           5.44539839e-01,  5.39180338e-01,  5.37910759e-01],
         [ 5.15340567e-01,  5.15650809e-01,  5.15915602e-01, ...,
           5.11486530e-01,  5.13205230e-01,  5.14603555e-01],
         [ 6.19197190e-01,  6.17705981e-01,  6.15579287e-01, ...,
           6.10567778e-01,  6.13886416e-01,  6.15758896e-01]],

        [[ 3.07214161e-01,  3.07687124e-01,  3.08225850e-01, ...,
           3.04641128e-01,  3.06042135e-01,  3.06481620e-01],
         [ 3.67408395e-01,  3.65119510e-01,  3.62561610e-01, ...,
           3.73642776e-01,  3.71497340e-01,  3.69482504e-01],
         [ 3.00429463e-01,  3.02731565e-01,  2.96060443e-01, ...,
           3.01417841e-01,  3.03042597e-01,  2.53353715e-01],
         ...,
         [ 7.00539947e-01,  6.93322122e-01,  6.60545826e-01, ...,
           7.04647899e-01,  7.03358054e-01,  7.02994466e-01],
         [ 6.63169026e-01,  6.70310259e-01,  6.76464438e-01, ...,
           6.33575916e-01,  6.44313693e-01,  6.54556155e-01],
         [ 6.59373204e-01,  6.61373138e-01,  6.63401484e-01, ...,
           6.48363630e-01,  6.52856827e-01,  6.55350685e-01]]]])
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
    height   float64 2.0
  * period   (period) object '1950-2022' '1951-2022' ... '2012-2022' '2013-2022'
Dimensions without coordinates: model

In [10]:
MMEM_forced_trend_da

<xarray.DataArray 'trend' (period: 64, lat: 90, lon: 180)>
array([[[0.18881608, 0.18898309, 0.18918949, ..., 0.18817149,
         0.18841581, 0.18859497],
        [0.18440655, 0.18460105, 0.1846545 , ..., 0.1837541 ,
         0.18397848, 0.18435883],
        [0.18489604, 0.18642983, 0.18697619, ..., 0.18284352,
         0.18383908, 0.18440724],
        ...,
        [0.52425964, 0.52478696, 0.52486412, ..., 0.52249855,
         0.52277282, 0.52318625],
        [0.52446837, 0.52475806, 0.52531051, ..., 0.52313233,
         0.52338769, 0.524149  ],
        [0.52651171, 0.52663279, 0.526775  , ..., 0.52601403,
         0.52594211, 0.52609338]],

       [[0.19232778, 0.19246359, 0.1926778 , ..., 0.19187616,
         0.19192187, 0.19196356],
        [0.18745357, 0.18770255, 0.18774533, ..., 0.18709479,
         0.18720061, 0.18747198],
        [0.18808036, 0.18903844, 0.18974691, ..., 0.18565961,
         0.18664069, 0.18744611],
...
        [0.97492203, 0.98410714, 0.99002859, ..., 0.95299589,
         0.95734712, 0.96479712],
        [0.97663177, 0.97306595, 0.97487868, ..., 0.97683907,
         0.97691685, 0.97864873],
        [1.00336453, 1.00551541, 1.00756725, ..., 0.99899285,
         1.00000232, 1.00125288]],

       [[0.35585205, 0.3543319 , 0.35601214, ..., 0.35287185,
         0.35341799, 0.35410446],
        [0.339385  , 0.3410461 , 0.34100892, ..., 0.33214978,
         0.3331085 , 0.33531283],
        [0.33807939, 0.3454594 , 0.34965359, ..., 0.33623345,
         0.33746261, 0.32656727],
        ...,
        [0.97674273, 0.98345784, 0.98311922, ..., 0.94805939,
         0.95804515, 0.97104155],
        [0.99726538, 1.00333696, 1.0097783 , ..., 0.98751942,
         0.99182387, 0.99533816],
        [1.0231158 , 1.02522263, 1.02744082, ..., 1.01429294,
         1.016748  , 1.01924111]]])
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
    height   float64 2.0
  * period   (period) object '1950-2022' '1951-2022' ... '2012-2022' '2013-2022'

In [13]:
# output the ensemble mean trend
import os
dir_out = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG3/MMLE/SMILE_forced/'
os.makedirs(dir_out, exist_ok=True)
MMEM_forced_trend_da.to_dataset(name='trend').to_netcdf(dir_out + 'MMLE_forced_MK_trend_1950-2022_sliding.nc')

### Calculation of the MMEM anomalies with each SMILEs anomalies

In [15]:
dir_in ='/work/mh0033/m301036/OBS_LPS_revision/docs/data/LE_data/ENS/'
MMEM_annual_ensemble_mean = xr.open_mfdataset(dir_in + 'MMEM_annual_ano_ensemble_mean_1950_2022_smbbCESM2_complement.nc',chunks={'lat': 10, 'lon': 10})

In [16]:
MMEM_annual_ensemble_mean

<xarray.Dataset>
Dimensions:  (lon: 180, lat: 90, year: 73)
Coordinates:
  * lon      (lon) float64 0.0 2.0 4.0 6.0 8.0 ... 350.0 352.0 354.0 356.0 358.0
  * lat      (lat) float64 -89.0 -87.0 -85.0 -83.0 -81.0 ... 83.0 85.0 87.0 89.0
  * year     (year) int64 1950 1951 1952 1953 1954 ... 2018 2019 2020 2021 2022
Data variables:
    tas      (year, lat, lon) float32 dask.array<chunksize=(73, 10, 10), meta=np.ndarray>

In [ ]:
# Input the MMEM annual mean SAT data
input_model = '/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Datafiles/'

CanESM_data   = xr.open_mfdataset(input_model + 'tas_CanESM5_annual_ano_1850_2022.nc',chunks = {'run':1})
IPSL_data     = xr.open_mfdataset(input_model + 'tas_IPSL_annual_ano_1850_2022.nc',chunks = {'run':1})
EC_Earth_data = xr.open_mfdataset(input_model + 'tas_EC_Earth_annual_ano_1850_2022.nc',chunks = {'run':1})
ACCESS_data   = xr.open_mfdataset(input_model + 'tas_ACCESS_annual_ano_1850_2022.nc',chunks = {'run':1})
MPI_ESM_data  = xr.open_mfdataset(input_model + 'tas_MPI_ESM_annual_ano_1850_2022.nc',chunks = {'run':1})
MIROC_data    = xr.open_mfdataset(input_model + 'tas_MIROC6_annual_ano_1850_2022.nc',chunks = {'run':1})
# MMEM_annual_data = xr.open_mfdataset(input_model + 'tas_MMEM_annual_anomalies_ds.nc',chunks = {'lat':10,'lon':10})

In [ ]:
# Extreact each SMILEs 1950-2022 data then output the ensemble mean anomalies
CanESM_ano = CanESM_data.tas.sel(year=slice('1950','2022')).squeeze()
IPSL_ano = IPSL_data.tas.sel(year=slice('1950','2022')).squeeze()
EC_Earth_ano = EC_Earth_data.tas.sel(year=slice('1950','2022')).squeeze()
ACCESS_ano = ACCESS_data.tas.sel(year=slice('1950','2022')).squeeze()
MPI_ESM_ano = MPI_ESM_data.tas.sel(year=slice('1950','2022')).squeeze()
MIROC_ano = MIROC_data.tas.sel(year=slice('1950','2022')).squeeze()

In [ ]:
CanESM_ano_mean = CanESM_ano.mean(dim='run')
IPSL_ano_mean = IPSL_ano.mean(dim='run')
EC_Earth_ano_mean = EC_Earth_ano.mean(dim='run')
ACCESS_ano_mean = ACCESS_ano.mean(dim='run')
MPI_ESM_ano_mean = MPI_ESM_ano.mean(dim='run')
MIROC_ano_mean = MIROC_ano.mean(dim='run')

In [ ]:
CanESM_ano_mean

In [ ]:
dir_output ='/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Supp_Figure1_Forced/data/'
CanESM_ano_mean.to_netcdf(dir_output + 'CanESM5_annual_ano_ensemble_mean_1950_2022.nc')
IPSL_ano_mean.to_netcdf(dir_output + 'IPSL_annual_ano_ensemble_mean_1950_2022.nc')
EC_Earth_ano_mean.to_netcdf(dir_output + 'EC_Earth_annual_ano_ensemble_mean_1950_2022.nc')
ACCESS_ano_mean.to_netcdf(dir_output + 'ACCESS_annual_ano_ensemble_mean_1950_2022.nc')
MPI_ESM_ano_mean.to_netcdf(dir_output + 'MPI_ESM_annual_ano_ensemble_mean_1950_2022.nc')
MIROC_ano_mean.to_netcdf(dir_output + 'MIROC6_annual_ano_ensemble_mean_1950_2022.nc')

In [ ]:
# Concatenate the data
MMEM_annual_data = xr.concat([CanESM_data,IPSL_data,EC_Earth_data,ACCESS_data,MPI_ESM_data,MIROC_data],dim='run')

In [ ]:
MMEM_annual_ensemble_mean = MMEM_annual_data.sel(year=slice(1950,2022)).mean(dim='run')

In [ ]:
MMEM_annual_ensemble_mean

In [ ]:
# # output the data
# dir_output ='/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Supp_Figure1_Forced/data/'
MMEM_annual_ensemble_mean.to_netcdf(dir_output + 'MMEM_annual_ano_ensemble_mean_1950_2022.nc')

### Calculate the 1950-2022 Model-ENS simulated trend patterns

In [ ]:
# separate the data into two sets: Pre-1950 and Post-1950
"""
    Calcualte the trend pattern for the forced variability on each grid point for the consective intervals starting from 
    1940-1949, 1935-1949, 1930-1949, 1925-1949, 1920-1949, 1915-1949, 1910-1949, 1905-1949, 1900-1949, 1895-1949, 1890-1949, 1885-1949, 
    1880-1949, 1875-1949, 1870-1949, 1865-1949, 1860-1949, 1855-1949, 1850-1949
    Put the data segemnet into the same dataset
"""
time_interval = {
    "10yr":(2013,2022),
    "20yr":(2003,2022),
    "30yr":(1993,2022),
    "40yr":(1983,2022),
    "50yr":(1973,2022),
    "60yr":(1963,2022),
    "70yr":(1953,2022)
}
start_year = 1950
end_year   = 2022

In [ ]:
# define the function to calculate the trend pattern for the forced variability on each grid point for the consective intervals
variable_name = ['10yr', '20yr', '30yr', '40yr', '50yr', '60yr', '70yr']
# define the function
def separate_data_into_intervals(data, start_year, end_year, time_interval):
    """
    This function is used to separate the data into different time intervals
    """
    # create a dictionary to store the data
    data_dict = {}
    for i in range(len(variable_name)):
        # calculate the start year and end year
        start_year = time_interval[variable_name[i]][0]
        end_year = time_interval[variable_name[i]][1]
        # select the data
        data_dict[variable_name[i]] = data.sel(year=slice(str(start_year), str(end_year)))
    return data_dict

In [ ]:
MMEM_annual_tas_dict = separate_data_into_intervals(MMEM_annual_ensemble_mean['tas'], start_year, end_year, time_interval)
MMEM_annual_tas_dict['10yr']

In [ ]:
MMEM_annual_tas_trend = {}
MMEM_annual_tas_p_value = {}

for i in range(len(variable_name)):
    data_var = MMEM_annual_tas_dict[variable_name[i]]
#     print(data_var)
    slope, p_values= xr.apply_ufunc(
        data_process.apply_mannkendall,
        data_var,
        input_core_dims=[["year"]],
        output_core_dims=[[], []],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float, float],
    )
    MMEM_annual_tas_trend[variable_name[i]] = slope
    MMEM_annual_tas_p_value[variable_name[i]] = p_values

In [ ]:
MMEM_annual_tas_trend_numpy = {}
MMEM_annual_tas_p_value_numpy = {}

for i in range(len(variable_name)):
    MMEM_annual_tas_trend_numpy[variable_name[i]] = MMEM_annual_tas_trend[variable_name[i]].values
    MMEM_annual_tas_p_value_numpy[variable_name[i]] = MMEM_annual_tas_p_value[variable_name[i]].values
    
MMEM_annual_tas_trend_numpy, MMEM_annual_tas_p_value_numpy

In [ ]:
MMEM_annual_tas_trend.items()

In [ ]:
MMEM_annual_tas_trend_da = {}

for interval, data in MMEM_annual_tas_trend.items():
    MMEM_annual_tas_trend_da[interval] = xr.DataArray(MMEM_annual_tas_trend_numpy[interval], 
                                                              dims = ['lat','lon'], 
                                                              coords = {'lat':MMEM_annual_tas_dict[variable_name[i]].lat.values, 
                                                                        'lon':MMEM_annual_tas_dict[variable_name[i]].lon.values})

In [ ]:
MMEM_annual_tas_p_value_da = {}

for interval, data in MMEM_annual_tas_p_value.items():
    MMEM_annual_tas_p_value_da[interval] = xr.DataArray(MMEM_annual_tas_p_value_numpy[interval], 
                                                              dims = ['lat','lon'], 
                                                              coords = {'lat':MMEM_annual_tas_dict[variable_name[i]].lat.values, 
                                                                        'lon':MMEM_annual_tas_dict[variable_name[i]].lon.values})

In [ ]:
# save the data into netcdf file
dir_output = '/work/mh0033/m301036/Land_surf_temp/analyses_1850_2100/Manuscript_visual_schematic/Disentangling_trend_analysis/Figure4/data/'
for interval, data in MMEM_annual_tas_trend_da.items():
    data.to_netcdf(dir_output + 'MMEM_annual_forced_' + interval + '_trend.nc')
    
for interval, data in MMEM_annual_tas_p_value.items():
    data.to_netcdf(dir_output + 'MMEM_annual_forced_' + interval + '_p_value.nc')

In [ ]:
# Check min, max of the trend data
for i in range(len(variable_name)):
    print(variable_name[i], MMEM_annual_tas_trend_da[variable_name[i]].min(), MMEM_annual_tas_trend_da[variable_name[i]].max())

### Plotting with the Robinson Projections

In [ ]:
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap

def plot_trend_with_significance(trend_data, lats, lons, p_values, GMST_p_values=None, levels=None, extend=None,cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """

    # Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
  
    # Determine significance mask (where p-values are less than 0.05)
    insignificance_mask = p_values >= 0.05
    
    # Plotting
    # contour_obj = ax.pcolormesh(lons, lats, trend_data,  cmap='RdBu_r',vmin=-5.0, vmax=5.0, transform=ccrs.PlateCarree(central_longitude=180), shading='auto')
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))

    # Plot significance masks with different hatches
    ax.contourf(lons, lats, insignificance_mask, levels=[0,0.05, 1.0],hatches=[None,'///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False,
                      color='gray', alpha=0.35, linestyle='--')

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 14}
    gl.ylabel_style = {'size': 14}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=18, pad=5.0)

    return contour_obj


In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

# intervals = [-0.2, -0.15, -0.1, -0.05, 0, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.9, 1.1, 1.3]
intervals = [-0.1, -0.075, -0.05, -0.025, 0, 0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 1.75, 2.0, 2.25, 2.5, 2.75]

# Normalizing the intervals to [0, 1]
min_interval = min(intervals)
max_interval = max(intervals)
normalized_intervals = [(val - min_interval) / (max_interval - min_interval) for val in intervals]

# cmap = mcolors.ListedColormap(palettable.scientific.diverging.Vik_20.mpl_colors)
cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

# Define the colors at each interval
colors = [(0.00784313725490196, 0.2, 0.4627450980392157, 1.0),
    (0.00784313725490196, 0.2, 0.4627450980392157, 1.0),
    (0.023529411764705882, 0.32941176470588235, 0.5450980392156862, 1.0),
    (0.023529411764705882, 0.32941176470588235, 0.5450980392156862, 1.0),
    (1.0, 1.0, 1.0, 1.0),
    (1.0, 0.9, 0.98, 1.0),
    (1.0, 0.8, 0.5, 1.0),
    (1.0, 0.803921568627451, 0.607843137254902, 1.0), 
    (1.0, 0.6000000000000001, 0.20000000000000018, 1.0),
    (1.0, 0.4039215686274509, 0.0, 1.0),
    (0.8999999999999999, 0.19999999999999996, 0.0, 1.0),
    (0.7470588235294118, 0.0, 0.0, 1.0), 
    (0.6000000000000001, 0.0, 0.0, 1.0),
    (0.44705882352941173, 0.0, 0.0, 1.0),
    (0.30000000000000004, 0.0, 0.0, 1.0),
    (0.14705882352941177, 0.0, 0.0, 1.0),
    (0.0, 0.0, 0.0, 1.0)]

# Creating a list of tuples with normalized positions and corresponding colors
color_list = list(zip(normalized_intervals, colors))

# Create the colormap
custom_cmap = LinearSegmentedColormap.from_list('my_custom_cmap', color_list)

# Create a normalization
norm = Normalize(vmin=min_interval, vmax=max_interval)

### Plot the Original, Forced, MMEM trend patterns

In [ ]:
# transform the trend data unit to degree per decade
for i in range(len(variable_name)):
    MMEM_annual_tas_trend_da[variable_name[i]] = MMEM_annual_tas_trend_da[variable_name[i]] * 10

In [ ]:
lat = MMEM_annual_tas_dict['10yr']['lat'].values
lon = MMEM_annual_tas_dict['10yr']['lon'].values
lat, lon 

titles = ["2013-2022(10yr)", "2003-2022(20yr)", "1993-2022(30yr)", "1983-2022(40yr)", "1973-2022(50yr)", "1963-2022(60yr)", "1953-2022(70yr)"]
titles_left = ["a.", "b.", "c.", "d.", "e.", "f.", "g."]

import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable

# Define the GridSpec
fig = plt.figure(figsize=(25, 15))
gs = gridspec.GridSpec(3, 3, height_ratios=[1, 1, 1], width_ratios=[1, 1, 1], wspace=0.01, hspace=0.01)

extend= 'max'
periods = ["10yr", "20yr", "30yr", "40yr", "50yr", "60yr", "70yr"]
for j, period in enumerate(periods):
    # Define the axes
    ax = fig.add_subplot(gs[j], projection=ccrs.Robinson(180))
    is_left = (j % 3 == 0)
    is_bottom_row = j >= (len(periods)//3)*3 
    
    trend_data = MMEM_annual_tas_trend_da[period]
    trend_with_cyclic, lon_with_cyclic = cutil.add_cyclic_point(trend_data, coord=lon)
    p_values = MMEM_annual_tas_p_value[period]
    p_values_with_cyclic, lon_with_cyclic = cutil.add_cyclic_point(p_values, coord=lon)
    levels = np.arange(-0.6, 0.65, 0.05)
    contour_obj = plot_trend_with_significance(trend_with_cyclic, lat, lon_with_cyclic, p_values_with_cyclic, 
                    GMST_p_values=None, levels=levels,extend=extend, cmap='twilight_shifted',
                    # cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors),
                    title=titles[j], ax=ax, show_xticks = is_bottom_row, show_yticks = is_left)
    ax.text(-0.03, 1.05, titles_left[j], fontsize=22,weight='bold', ha='center', va='center', rotation='horizontal', transform=ax.transAxes)

# Add horizontal colorbars
cbar_ax = fig.add_axes([0.43, 0.2, 0.5, 0.025])
cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation='horizontal')
cbar.ax.tick_params(labelsize=18)
cbar.set_label('Annual SAT Trend (°C/decade)', fontsize=22)

plt.tight_layout()
fig.savefig('MMEM_simulated_trend_Pattern_variations.png', dpi=300, bbox_inches='tight')

plt.show()

In [14]:
# client.close()
# scluster.close()